# Liquidation Strategy Walkthrough

This walkthrough follows the same service layer used by the Streamlit application for Liquidity Management Tools Calibration. It is written for fund-risk review: the code cells call application services, while the markdown explains the business meaning of the returned results.

The walkthrough covers both the single-period calibration workflow and the implemented 12-month redemption-path workflow. It does not reimplement liquidation, market stress, redemption demand, LMT assessment, or path calculations inside notebook cells.

## Setup

The setup imports display helpers, pandas, and the application service functions. The calculation boundary is `lmt_calibration.services`, matching the app workflow.

In [1]:
from decimal import Decimal

import pandas as pd
from _notebook_helpers import days, gross_sales, money, print_profile, rate, yes_no
from _notebook_setup import SAMPLE_DATA_DIR, configure_display

from lmt_calibration.services import (
    build_historical_result_rows,
    build_scenario_matrix_outcome,
    build_t0_liquidity_profile_rows,
    fund_positions,
    load_app_sample_data,
    run_sample_redemption_path,
    run_scenario_across_market_conditions,
    run_selected_sample_scenario,
)

configure_display()

## Load Application Sample Data

The sample files are loaded through the same application service used by Streamlit. This keeps the walkthrough aligned with loader validation, domain-object construction, and service-level lookups.

In [2]:
inputs = load_app_sample_data(SAMPLE_DATA_DIR)

sample_input_counts = [
    ("funds", len(inputs.funds)),
    ("positions", len(inputs.positions)),
    ("investor_classes", len(inputs.investor_classes)),
    ("redemption_scenarios", len(inputs.redemption_scenarios)),
    ("market_stresses", len(inputs.market_stresses)),
    ("liquidity_stresses", len(inputs.liquidity_stresses)),
    ("scenario_definitions", len(inputs.scenario_definitions)),
    ("lmt_parameters", len(inputs.lmt_parameters)),
    ("liquidation_strategies", len(inputs.liquidation_strategies)),
]

pd.DataFrame(sample_input_counts, columns=["dataset", "records"])

,dataset,records
0,funds,1
1,positions,9
2,investor_classes,5
3,redemption_scenarios,3
4,market_stresses,4
5,liquidity_stresses,3
6,scenario_definitions,6
7,lmt_parameters,2
8,liquidation_strategies,4


# Part 1 - Single-Period Application Workflow

The single-period workflow is the current calibration matrix workflow. A selected fund, redemption scenario, market stress, liquidity stress, liquidation strategy, and LMT parameter set are assembled by the service layer. The service applies the methodology and returns a run object containing the inputs, liquidation result, execution-cost context, and simulated activation assessment.

In [3]:
selected_fund_id = "lux_dynamic_allocation"
selected_redemption_scenario_id = "severe_platform_outflow"
selected_strategy_id = "partial_cash_then_pro_rata"

single_period_run = run_selected_sample_scenario(
    inputs,
    fund_id=selected_fund_id,
    strategy_id=selected_strategy_id,
    redemption_scenario_id_override=selected_redemption_scenario_id,
)
single_period_result = single_period_run.result

scenario_profile = {
    "fund_id": single_period_run.fund.fund_id,
    "fund_name": single_period_run.fund.fund_name,
    "redemption_scenario": single_period_run.redemption.name,
    "market_stress": single_period_run.market_stress.name,
    "liquidity_stress": single_period_run.liquidity_stress.name,
    "liquidation_strategy": single_period_run.strategy.name,
    "strategy_type": single_period_run.strategy.strategy_type.value,
    "lmt_parameter_set": single_period_run.parameters.parameter_set_id,
}

print_profile(scenario_profile)

              fund_id:      lux_dynamic_allocation
            fund_name: Lux Dynamic Allocation Fund
  redemption_scenario:     severe_platform_outflow
        market_stress:    normal_market_conditions
     liquidity_stress:   normal_liquidity_capacity
 liquidation_strategy:  partial_cash_then_pro_rata
        strategy_type:                      hybrid
    lmt_parameter_set:         board_approved_base


## Fund And Portfolio Snapshot

The fund snapshot and position set are the opening point for the application workflow. Cash, listed equities, listed ETFs, reverse repos, and repo financing exposures are intentionally shown separately because the liquidation engine treats their liquidity differently.

In [4]:
fund = single_period_run.fund
positions = fund_positions(inputs, fund)
liquidity_profile_rows = build_t0_liquidity_profile_rows(positions)

fund_profile = {
    "as_of_date": str(fund.as_of_date),
    "base_currency": fund.base_currency,
    "nav": money(fund.nav),
    "dealing_frequency": fund.dealing_frequency,
    "redemption_notice_days": days(fund.redemption_notice_days),
    "redemption_settlement_days": days(fund.redemption_settlement_days),
}

print_profile(fund_profile)

                 as_of_date:     2026-06-30
              base_currency:            EUR
                        nav: 100,000,000.00
          dealing_frequency:          daily
     redemption_notice_days:              1
 redemption_settlement_days:              3


In [5]:
pd.DataFrame(
    [
        {
            "instrument_name": position.instrument_name,
            "asset_group": position.asset_group.value,
            "market_value": money(position.market_value),
            "notional_amount": money(position.notional_amount),
            "base_haircut_rate": rate(position.base_haircut_rate),
            "base_liquidity_capacity_rate": rate(position.base_liquidity_capacity_rate),
            "settlement_days": days(position.settlement_days),
            "maturity_days": days(position.maturity_days),
        }
        for position in positions
    ]
)

,instrument_name,asset_group,market_value,notional_amount,base_haircut_rate,base_liquidity_capacity_rate,settlement_days,maturity_days
0,EUR Operating Cash,cash,"12,000,000.00",,0.00%,100.00%,0,
1,SAP SE Ordinary Shares,listed_equity,"15,000,000.00",,5.00%,20.00%,2,
2,ASML Holding NV Ordinary Shares,listed_equity,"12,000,000.00",,6.00%,18.00%,2,
3,LVMH Moet Hennessy Louis Vuitton SE Ordinary S...,listed_equity,"8,000,000.00",,6.00%,18.00%,2,
4,iShares Core MSCI World UCITS ETF,listed_etf,"18,000,000.00",,4.00%,35.00%,2,
5,Xtrackers Euro Stoxx 50 UCITS ETF,listed_etf,"10,000,000.00",,4.00%,40.00%,2,
6,EUR Overnight Reverse Repo BNP Paribas Synthetic,reverse_repo,"15,000,000.00",,1.00%,100.00%,1,1
7,EUR One Week Reverse Repo Societe Generale Syn...,reverse_repo,"10,000,000.00",,1.00%,90.00%,1,7
8,EUR Repo Financing Obligation Synthetic,repo_financing,,"5,000,000.00",0.00%,0.00%,1,


## Liquidity Profile At The Opening Date

The service prepares the same liquidity-bucket rows used by the application. The rows separate NAV by liquidity bucket and estimate liquid resources available under the current position attributes.

In [6]:
pd.DataFrame(
    [
        {
            "liquidity_bucket": row["liquidity_bucket"],
            "nav_amount": money(row["nav_amount"]),
            "liquid_resources": money(row["liquid_resources"]),
            "unavailable_nav": money(row["unavailable_nav"]),
        }
        for row in liquidity_profile_rows
    ]
)

,liquidity_bucket,nav_amount,liquid_resources,unavailable_nav
0,Cash,"12,000,000.00","12,000,000.00",0.00
1,0-7 days,"78,000,000.00","31,900,000.00","46,100,000.00"
2,8-30 days,"10,000,000.00","9,000,000.00","1,000,000.00"
3,>30 days / constrained,0.00,0.00,0.00


## Redemption Pressure And Liquidation Result

The service computes the redemption amount from investor-class assumptions and the selected redemption scenario. The liquidation result then shows how the selected strategy uses cash and asset sales to meet that cash need subject to the minimum buffer, capacity, haircut, settlement, and maturity constraints.

In [7]:
liquidation_summary = {
    "total_redemption_amount": money(single_period_result.total_redemption_amount),
    "redemption_rate": rate(single_period_run.redemption_rate),
    "cash_used": money(single_period_result.cash_used),
    "gross_sales": money(gross_sales(single_period_result)),
    "post_haircut_cash_raised": money(single_period_result.total_post_haircut_cash_raised),
    "dilution_amount": money(single_period_result.dilution_amount),
    "dilution_rate": rate(single_period_result.dilution_rate),
    "shortfall": money(single_period_result.shortfall),
    "remaining_liquid_buffer_rate": rate(single_period_result.remaining_liquid_buffer_rate),
    "minimum_cash_buffer_preserved": yes_no(single_period_result.minimum_cash_buffer_preserved),
}

print_profile(liquidation_summary)

       total_redemption_amount: 17,250,000.00
               redemption_rate:        17.25%
                     cash_used:  3,500,000.00
                   gross_sales:  8,770,940.17
      post_haircut_cash_raised:  8,288,230.77
               dilution_amount:    482,709.40
                 dilution_rate:         0.48%
                     shortfall:  5,461,769.23
  remaining_liquid_buffer_rate:        25.17%
 minimum_cash_buffer_preserved:           yes


In [8]:
pd.DataFrame(
    [
        {
            "position_id": asset.position_id,
            "asset_group": asset.asset_group.value,
            "gross_sale_amount": money(asset.gross_sale_amount),
            "post_haircut_cash_raised": money(asset.post_haircut_cash_raised),
            "haircut_cost": money(asset.haircut_cost),
        }
        for asset in single_period_result.assets_liquidated
    ]
)

,position_id,asset_group,gross_sale_amount,post_haircut_cash_raised,haircut_cost
0,asml_equity_position,listed_equity,"648,000.00","576,720.00","71,280.00"
1,euro_stoxx_etf_position,listed_etf,"1,600,000.00","1,504,000.00","96,000.00"
2,lvmh_equity_position,listed_equity,"432,000.00","384,480.00","47,520.00"
3,msci_world_etf_position,listed_etf,"2,520,000.00","2,368,800.00","151,200.00"
4,overnight_reverse_repo_bnp,reverse_repo,"2,670,940.17","2,644,230.77","26,709.40"
5,sap_equity_position,listed_equity,"900,000.00","810,000.00","90,000.00"


## Activation Assessment

The application labels LMT outputs as simulated activation assessment. These diagnostics support calibration review; they do not decide whether a fund manager activates an LMT.

In [9]:
activation = single_period_run.lmt_activation
activation_summary = {
    "swing_pricing": yes_no(activation.swing_activated),
    "redemption_gate": yes_no(activation.gate_activated),
    "liquidity_buffer_breach": yes_no(activation.buffer_breached),
    "calibration_adequacy": activation.calibration_adequacy,
    "nav_after_redemption_before_lmt": money(activation.nav_after_redemption_before_lmt),
    "applied_swing_factor_rate": rate(activation.applied_swing_factor_rate),
    "applied_cost_recovery_amount": money(activation.applied_cost_recovery_amount),
    "redemption_paid_amount": money(activation.redemption_paid_amount),
    "redemption_deferred_amount": money(activation.redemption_deferred_amount),
    "current_post_lmt_nav": money(activation.current_post_lmt_nav),
    "remaining_liquid_buffer_rate": rate(activation.remaining_liquid_buffer_rate),
}

print_profile(activation_summary)

                   swing_pricing:                         yes
                 redemption_gate:                         yes
         liquidity_buffer_breach:                          no
            calibration_adequacy: CalibrationAdequacy.ALIGNED
 nav_after_redemption_before_lmt:               82,267,290.60
       applied_swing_factor_rate:                       0.10%
    applied_cost_recovery_amount:                   17,767.50
          redemption_paid_amount:               10,000,000.00
      redemption_deferred_amount:                7,250,000.00
            current_post_lmt_nav:               89,535,058.10
    remaining_liquid_buffer_rate:                      23.13%


## Scenario Matrix View

The matrix view runs the selected redemption and liquidation strategy across the available market conditions. This is the same comparison boundary used by the application: each column is an independent service run.

In [10]:
matrix_runs = run_scenario_across_market_conditions(
    inputs,
    fund_id=selected_fund_id,
    strategy_id=selected_strategy_id,
    redemption_scenario_id_override=selected_redemption_scenario_id,
)

matrix_rows = []
for run in matrix_runs:
    outcome = build_scenario_matrix_outcome(run)
    matrix_rows.append(
        {
            "market_stress": run.market_stress.name,
            "current_pre_lmt_nav": money(run.current_pre_lmt_nav),
            "redemption_amount": money(run.redemption_amount),
            "realised_liquidity_cost": money(outcome.realised_liquidity_cost),
            "current_post_lmt_nav": money(outcome.current_post_lmt_nav),
            "remaining_liquid_buffer_rate_after_lmt": rate(
                outcome.remaining_liquid_buffer_rate_after_lmt
            ),
            "calibration_adequacy": run.lmt_activation.calibration_adequacy,
        }
    )

pd.DataFrame(matrix_rows)

,market_stress,current_pre_lmt_nav,redemption_amount,realised_liquidity_cost,current_post_lmt_nav,remaining_liquid_buffer_rate_after_lmt,calibration_adequacy
0,normal_market_conditions,"100,000,000.00","17,250,000.00","482,709.40","89,535,058.10",23.13%,Aligned
1,moderate_market_stress,"96,850,000.00","17,250,000.00","461,033.44","86,721,439.19",23.75%,Aligned
2,severe_market_stress,"92,440,000.00","17,250,000.00","430,856.00","82,782,170.07",24.67%,Aligned
3,historical_crisis_2008,"77,950,000.00","17,250,000.00","333,635.63","69,836,567.39",28.16%,Aligned


## Historical Market Stress Reference Rows

Historical market stress rows are reference context. They are loaded by the same application service and displayed as context rather than silently changing the active workflow.

In [11]:
pd.DataFrame(build_historical_result_rows(inputs, single_period_run))

,scenario_id,scenario,period,holding_period_days,cash_used,post_haircut_cash_raised,shortfall,dilution,remaining_buffer,status
0,historical_2008_financial_crisis,2008 Financial Crisis,September - December 2008,20,3500000.0000,8288230.769230769230769230769,5461769.23076923076923076923,482709.4017094017094017094017,0.2312587903616331594337695137,Context only
1,historical_2020_covid,2020 COVID-19 Crash,February - March 2020,20,3500000.0000,8288230.769230769230769230769,5461769.23076923076923076923,482709.4017094017094017094017,0.2312587903616331594337695137,Context only
2,historical_2022_rate_inflation,2022 Rate and Inflation Shock,January - December 2022,20,3500000.0000,8288230.769230769230769230769,5461769.23076923076923076923,482709.4017094017094017094017,0.2312587903616331594337695137,Context only


# Part 2 - 12-Month Redemption Path Workflow

The 12-month path uses the implemented multi-period service. Normal months draw investor-class redemption rates from the Beta methodology. Selected redemption-stress months replace the sampled normal rate with each class's stress redemption rate. Threshold signals are separated from applied LMT governance assumptions. This walkthrough uses the signal-linked option for swing pricing and gates; suspension remains explicitly selected. Deferred backlog is carried forward by investor class, and liquidation is calculated only for paid redemption.

In [12]:
path_run = run_sample_redemption_path(
    inputs,
    fund_id=selected_fund_id,
    strategy_id=selected_strategy_id,
    redemption_scenario_id=selected_redemption_scenario_id,
    lmt_parameters_override=None,
    stress_months=(1,),
    random_seed=42,
    market_stress_id=None,
    market_stress_month=None,
    behavioural_feedback_multiplier=Decimal("1"),
    market_contagion_liquidity_cost_multiplier=Decimal("1"),
    apply_lmts_in_all_signal_months=True,
)

pd.DataFrame(path_run.configuration_rows)

,setting,value
0,Scenario,streamlit_selected_configuration_redemption_path
1,Redemption scenario,severe_platform_outflow
2,Redemption-stress months,1
3,LMT application mode,All signal months
4,Applied swing pricing months,"1, 2, 3, 4, 6, 7, 8, 10, 11, 12"
5,Applied gate months,"1, 2"
6,Applied suspension months,None
7,Market stress scenario,No market stress
8,Liquidation strategy,partial_cash_then_pro_rata
9,Random seed,42


## Monthly Redemption Path Summary

The monthly summary shows the main state variables carried through the path: new demand, effective demand, paid redemption, deferred redemption, backlog, NAV, cash, liquid resources, threshold signals, and applied LMT outcomes.

In [13]:
pd.DataFrame(
    [
        {
            "month": row["month"],
            "new_redemption_demand": money(row["new_redemption_demand"]),
            "effective_redemption_demand": money(row["effective_redemption_demand"]),
            "paid_redemption": money(row["paid_redemption"]),
            "deferred_redemption": money(row["deferred_redemption"]),
            "cumulative_backlog": money(row["cumulative_backlog"]),
            "closing_nav": money(row["closing_nav"]),
            "closing_cash": money(row["closing_cash"]),
            "liquid_resources": money(row["remaining_liquid_resources"]),
            "priority_outcome": row["priority_outcome"],
        }
        for row in path_run.monthly_rows
    ]
)

,month,new_redemption_demand,effective_redemption_demand,paid_redemption,deferred_redemption,cumulative_backlog,closing_nav,closing_cash,liquid_resources,priority_outcome
0,1,"17,250,000.00","17,250,000.00","10,000,000.00","7,250,000.00","7,250,000.00","90,000,000.00","27,000,000.00","32,644,000.00",redemption_gate
1,2,"2,737,012.40","9,987,012.40","9,000,000.00","987,012.40","987,012.40","81,000,000.00","18,000,000.00","23,644,000.00",redemption_gate
2,3,"1,795,586.28","2,782,598.68","2,782,598.68",0.00,0.00,"78,217,401.32","15,217,401.32","20,861,401.32",swing_pricing
3,4,"2,217,777.16","2,217,777.16","2,217,777.16",0.00,0.00,"75,999,624.16","12,999,624.16","18,643,624.16",swing_pricing
4,5,"1,096,289.68","1,096,289.68","1,096,289.68",0.00,0.00,"74,903,334.47","11,903,334.47","17,547,334.47",none
5,6,"1,312,167.25","1,312,167.25","1,312,167.25",0.00,0.00,"73,591,167.22","10,591,167.22","16,235,167.22",swing_pricing
6,7,"1,283,774.79","1,283,774.79","1,283,774.79",0.00,0.00,"72,307,392.43","9,307,392.43","14,951,392.43",swing_pricing
7,8,"1,323,585.32","1,323,585.32","1,323,585.32",0.00,0.00,"70,983,807.11","7,983,807.11","13,627,807.11",swing_pricing
8,9,"1,051,543.84","1,051,543.84","1,051,543.84",0.00,0.00,"69,932,263.27","6,932,263.27","12,576,263.27",none
9,10,"1,424,714.22","1,424,714.22","1,424,714.22",0.00,0.00,"68,507,549.05","5,507,549.05","11,151,549.05",swing_pricing


## Investor-Class Demand And Backlog

Investor-class rows help reviewers see which class drives demand and how paid and deferred amounts are allocated. Backlog is not multiplied again by behavioural feedback; it is carried mechanically until paid or deferred again. Market contagion increases incremental realised execution cost in the month after the one-time market stress event, reducing net proceeds and the path NAV without changing redemption demand.

In [14]:
pd.DataFrame(
    [
        {
            "month": row["month"],
            "client_class": row["client_class"],
            "new_redemption_demand": money(row["new_redemption_demand"]),
            "opening_backlog": money(row["opening_backlog"]),
            "effective_redemption_demand": money(row["effective_redemption_demand"]),
            "paid_redemption": money(row["paid_redemption"]),
            "deferred_redemption": money(row["deferred_redemption"]),
            "redemption_rate": rate(row["redemption_rate"]),
            "behavioural_feedback_source_outcome": row["behavioural_feedback_source_outcome"],
        }
        for row in path_run.investor_rows
    ]
)

,month,client_class,new_redemption_demand,opening_backlog,effective_redemption_demand,paid_redemption,deferred_redemption,redemption_rate,behavioural_feedback_source_outcome
0,1,fund_of_funds,"2,250,000.00",0.00,"2,250,000.00","1,304,347.83","945,652.17",15.00%,none
1,1,institutional,"4,500,000.00",0.00,"4,500,000.00","2,608,695.65","1,891,304.35",18.00%,none
2,1,platform,"6,750,000.00",0.00,"6,750,000.00","3,913,043.48","2,836,956.52",27.00%,none
3,1,retail,"3,600,000.00",0.00,"3,600,000.00","2,086,956.52","1,513,043.48",12.00%,none
4,1,seed_capital,"150,000.00",0.00,"150,000.00","86,956.52","63,043.48",3.00%,none
5,2,fund_of_funds,"224,038.03","945,652.17","1,169,690.21","1,054,090.20","115,600.01",1.76%,redemption_gate
6,2,institutional,"757,771.08","1,891,304.35","2,649,075.43","2,387,268.38","261,807.06",3.70%,redemption_gate
7,2,platform,"499,415.56","2,836,956.52","3,336,372.08","3,006,639.77","329,732.31",2.74%,redemption_gate
8,2,retail,"1,255,787.73","1,513,043.48","2,768,831.21","2,495,188.74","273,642.47",4.76%,redemption_gate
9,2,seed_capital,0.00,"63,043.48","63,043.48","56,812.92","6,230.56",0.00%,redemption_gate


## Monthly Activation Assessment

The path records threshold signals separately from applied swing pricing, gates, and suspension assumptions. Signal-linked mode applies swing pricing and gates in signal months, while suspension remains an explicitly selected governance scenario.

In [15]:
pd.DataFrame(
    [
        {
            "month": row["month"],
            "swing_signal": row["swing_signal"],
            "swing_applied": row["swing_applied"],
            "gate_signal": row["gate_signal"],
            "gate_applied": row["gate_applied"],
            "liquidity_buffer_breach": row["liquidity_buffer_breach"],
            "suspension_applied": row["suspension_applied"],
            "priority_outcome": row["priority_outcome"],
            "paid_redemption": money(row["paid_redemption"]),
            "deferred_redemption": money(row["deferred_redemption"]),
            "swing_recovery": money(row["swing_recovery"]),
        }
        for row in path_run.lmt_timeline_rows
    ]
)

,month,swing_signal,swing_applied,gate_signal,gate_applied,liquidity_buffer_breach,suspension_applied,priority_outcome,paid_redemption,deferred_redemption,swing_recovery
0,1,True,True,True,True,False,False,redemption_gate,"10,000,000.00","7,250,000.00",0.00
1,2,True,True,True,True,False,False,redemption_gate,"9,000,000.00","987,012.40",0.00
2,3,True,True,False,False,False,False,swing_pricing,"2,782,598.68",0.00,0.00
3,4,True,True,False,False,False,False,swing_pricing,"2,217,777.16",0.00,0.00
4,5,False,False,False,False,False,False,none,"1,096,289.68",0.00,0.00
5,6,True,True,False,False,False,False,swing_pricing,"1,312,167.25",0.00,0.00
6,7,True,True,False,False,False,False,swing_pricing,"1,283,774.79",0.00,0.00
7,8,True,True,False,False,False,False,swing_pricing,"1,323,585.32",0.00,0.00
8,9,False,False,False,False,False,False,none,"1,051,543.84",0.00,0.00
9,10,True,True,False,False,False,False,swing_pricing,"1,424,714.22",0.00,0.00


## Notes

This walkthrough intentionally calls the same service layer as the application. If a service output looks wrong, the fix belongs in the domain, engine, service, data, or methodology documentation rather than in notebook-local helper code.

The separate [liquidation strategy inspection notebook](liquidation_strategy_inspection.ipynb) remains available as a low-level diagnostic aid, but it still contains local inspection helpers and should be reviewed separately if the project wants every notebook to use only application services.